# Data Read & Process

In [0]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("DataProcessing").getOrCreate()

In [0]:
df1 = spark.read.format("csv").option("header","true").load("/Workspace/Users/vivek223singh@gmail.com/dataEngineering/customers10mb.csv")

In [0]:
df1.show(5) 
df1.printSchema()

In [0]:
from pyspark.sql.functions import * 
df1 = df1.withColumn("registration_date", to_date(col("registration_date"),"yyyy-MM-dd"))\
    .withColumn("is_active",col("is_active").cast("boolean"))

In [0]:
df1.printSchema()

In [0]:
df1 = df1.fillna({"country":"Unknown","city":"Unknown","state":"Unknown"})

In [0]:
df1 = df1.withColumn("registration_year",year(col("registration_date")))\
    .withColumn("registration_month",month(col("registration_date")))
    

In [0]:
df1.show(6)

In [0]:
unique_city = df1.select(countDistinct("city")).collect()
unique_city[0][0]

unique_state = df1.select(countDistinct("state")).collect()
unique_state[0][0]

unique_country = df1.select(countDistinct("country")).collect()
unique_country[0][0]

In [0]:
df1.groupBy("city").count().orderBy(col("Count").desc()).show()

In [0]:
df1.groupBy("city","state").count().orderBy(col("count").desc()).show()

In [0]:
#Pivot Table
df1.groupBy("city").pivot("is_active").count().show()

In [0]:
df1.show(7)

In [0]:
df1.filter(col("registration_date")>= lit("2023-09-09")).show()

In [0]:
from pyspark.sql.window import Window
window_spec = Window.partitionBy("state").orderBy(col("registration_date").desc())

df1.filter(col("registration_date")>="2023-09-09")\
    .withColumn("rank",rank().over(window_spec))\
    .select("name","state","rank","registration_date").show()

In [0]:
df1.groupBy("city").agg(
    min("registration_date").alias("oldest customer"),
    max("registration_date").alias("newest customer")
).show()


In [0]:
#Serveless doesn't support file write so use this instead
df1.write.mode("overwrite").saveAsTable("processed_data")

#check
spark.sql("select * from processed_data").show()


#To write as parquet
#store_path="/Filestore/tables/processed_data"
# df1.write.mode("overwrite").parquet(store_path)